# Assignment 09 — Kernel Matrices (100 points)

**Unit**: AI 200 — Mathematical Foundations for AI  
**Competition alignment**: USAAIO 2026 Round 1 & Round 2  

---

## Background

A kernel function $k(\mathbf{x}, \mathbf{y})$ computes an inner product in a (possibly infinite-dimensional) feature space without explicitly constructing the feature map. The Gram matrix $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$ must be positive semi-definite (PSD) for $k$ to be a valid kernel (Mercer's theorem). Common kernels: linear ($\mathbf{x}^\top\mathbf{y}$), polynomial ($(\mathbf{x}^\top\mathbf{y} + c)^d$), and RBF ($\exp(-\|\mathbf{x}-\mathbf{y}\|^2/2\sigma^2)$). Kernel PCA performs nonlinear dimensionality reduction by operating in feature space via the kernel trick.

## Notation

| Symbol | Meaning |
|--------|---------|
| $K$ | Gram (kernel) matrix, shape $(N, N)$ |
| $k(\mathbf{x},\mathbf{y})$ | kernel function |
| $\phi(\mathbf{x})$ | explicit feature map |
| $\sigma$ | RBF bandwidth |
| $\tilde{K}$ | centered kernel matrix |

In [ ]:
# DO NOT MAKE ANY CHANGE IN THIS CELL
import numpy as np
np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)

> **WARNING**: Do not import any additional libraries. No `sklearn.metrics.pairwise` for kernel computation (implement from scratch). You may use `np.linalg.eigh` for PCA steps.

---

## Part 1 (25 points, coding task)

**Kernel Functions**

Implement three kernel functions and construct their Gram matrices:
1. **Linear**: $k(\mathbf{x}, \mathbf{y}) = \mathbf{x}^\top \mathbf{y}$
2. **Polynomial**: $k(\mathbf{x}, \mathbf{y}) = (\mathbf{x}^\top \mathbf{y} + c)^d$
3. **RBF (Gaussian)**: $k(\mathbf{x}, \mathbf{y}) = \exp(-\|\mathbf{x} - \mathbf{y}\|^2 / 2\sigma^2)$

The RBF kernel must be computed **without loops** using the identity $\|\mathbf{x}-\mathbf{y}\|^2 = \|\mathbf{x}\|^2 + \|\mathbf{y}\|^2 - 2\mathbf{x}^\top\mathbf{y}$.

*Reasoning is not required.*

In [ ]:
def linear_kernel(X: np.ndarray) -> np.ndarray:
    """Compute linear kernel matrix K = X X^T.
    Args: X: shape (N, d)
    Returns: K: shape (N, N)
    """
    ### WRITE YOUR SOLUTION HERE ###
    pass

def polynomial_kernel(X: np.ndarray, degree: int = 3, c: float = 1.0) -> np.ndarray:
    """Compute polynomial kernel matrix K_ij = (x_i^T x_j + c)^d.
    Args: X: shape (N, d), degree: polynomial degree, c: offset
    Returns: K: shape (N, N)
    """
    ### WRITE YOUR SOLUTION HERE ###
    pass

def rbf_kernel(X: np.ndarray, sigma: float = 1.0) -> np.ndarray:
    """Compute RBF kernel matrix. No loops!
    Args: X: shape (N, d), sigma: bandwidth
    Returns: K: shape (N, N)
    Hint: ||x-y||^2 = ||x||^2 + ||y||^2 - 2 x^T y
    """
    ### WRITE YOUR SOLUTION HERE ###
    pass

""" END OF THIS PART """

In [ ]:
X = np.random.randn(50, 3)  # (50, 3)

K_linear = linear_kernel(X)             # (50, 50)
K_poly = polynomial_kernel(X, degree=3) # (50, 50)
K_rbf = rbf_kernel(X, sigma=1.0)        # (50, 50)

print(f"Linear kernel shape: {K_linear.shape}")
print(f"Polynomial kernel shape: {K_poly.shape}")
print(f"RBF kernel shape: {K_rbf.shape}")

---

A valid kernel matrix must be PSD: all eigenvalues $\geq 0$. The sum of two PSD matrices is PSD. The Hadamard (element-wise) product of two PSD matrices is also PSD (Schur product theorem). However, the ordinary matrix product of two PSD matrices is NOT necessarily PSD.

---

## Part 2 (25 points, mixed task)

**Verify PSD Property**

**(a)** (15 points, coding) Implement `verify_psd` that checks symmetry and eigenvalue non-negativity. Verify for all three kernels, their sum, and their Hadamard product. *Reasoning is not required.*

**(b)** (10 points, non-coding) Explain why the RBF kernel matrix is always PSD. (Hint: the RBF kernel can be written as $k(\mathbf{x},\mathbf{y}) = \exp(-\|\mathbf{x}\|^2/2\sigma^2) \cdot \exp(\mathbf{x}^\top\mathbf{y}/\sigma^2) \cdot \exp(-\|\mathbf{y}\|^2/2\sigma^2)$. Use the Taylor expansion of $\exp$ and the closure of PSD kernels under products and limits.) *Reasoning is required.*

In [ ]:
# Part 2a
def verify_psd(K: np.ndarray, name: str) -> bool:
    """Verify a matrix is PSD by checking eigenvalues.
    
    Args:
        K: shape (N, N) - kernel matrix
        name: string label
    
    Returns:
        is_psd: bool
    """
    ### WRITE YOUR SOLUTION HERE ###
    # 1. Check symmetry: K == K^T
    # 2. Compute eigenvalues
    # 3. Check all >= -epsilon (numerical tolerance)
    # 4. Print results
    pass

In [ ]:
verify_psd(K_linear, "Linear")
verify_psd(K_poly, "Polynomial")
verify_psd(K_rbf, "RBF")

K_sum = K_linear + K_rbf
verify_psd(K_sum, "Linear + RBF")

K_hadamard = K_linear * K_rbf  # element-wise
verify_psd(K_hadamard, "Linear * RBF (Hadamard)")

### WRITE YOUR SOLUTION HERE (Part 2b) ###



""" END OF THIS PART """

---

For the degree-2 polynomial kernel with $c=0$: $k(\mathbf{x},\mathbf{y}) = (\mathbf{x}^\top\mathbf{y})^2$. For 2D input $\mathbf{x} = [x_1, x_2]^\top$, the explicit feature map is $\phi(\mathbf{x}) = [x_1^2, \sqrt{2}x_1x_2, x_2^2]^\top$, and $k(\mathbf{x},\mathbf{y}) = \phi(\mathbf{x})^\top\phi(\mathbf{y})$.

---

## Part 3 (20 points, coding task)

**Feature Map Connection**

Implement the explicit feature map $\phi$ for the degree-2 polynomial kernel ($c=0$, 2D input). Verify that the Gram matrix $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$ equals $\Phi\Phi^\top$ where $\Phi$ has rows $\phi(\mathbf{x}_i)^\top$.

*Reasoning is not required.*

In [ ]:
def poly2_feature_map(X: np.ndarray) -> np.ndarray:
    """Explicit feature map for degree-2 polynomial kernel (c=0, 2D input).
    
    Args:
        X: shape (N, 2) - 2D input data
    
    Returns:
        Phi: shape (N, 3) - mapped features [x1^2, sqrt(2)*x1*x2, x2^2]
    """
    ### WRITE YOUR SOLUTION HERE ###
    pass

""" END OF THIS PART """

In [ ]:
X_2d = np.random.randn(30, 2)  # (30, 2)

K_poly2 = polynomial_kernel(X_2d, degree=2, c=0.0)  # (30, 30)

Phi = poly2_feature_map(X_2d)   # (30, 3)
K_feature = Phi @ Phi.T         # (30, 30)

print(f"Kernel matrix shape:         {K_poly2.shape}")
print(f"Feature map product shape:   {K_feature.shape}")
print(f"Match: {np.allclose(K_poly2, K_feature)}")
print(f"Max difference: {np.max(np.abs(K_poly2 - K_feature)):.2e}")

---

Kernel PCA performs PCA in feature space using only the kernel matrix. The algorithm: (1) compute $K$, (2) center it as $\tilde{K} = K - \frac{1}{N}\mathbf{1}K - \frac{1}{N}K\mathbf{1} + \frac{1}{N^2}\mathbf{1}K\mathbf{1}$, (3) eigendecompose $\tilde{K}$, and (4) project using the top eigenvectors scaled by $1/\sqrt{\lambda_i}$.

---

## Part 4 (30 points, coding task)

**Kernel PCA**

Implement kernel PCA and apply it to concentric circles data. Show that linear PCA fails to separate the circles but kernel PCA with RBF kernel succeeds.

*Reasoning is not required.*

In [ ]:
def kernel_pca(K: np.ndarray, k: int) -> np.ndarray:
    """Kernel PCA.
    
    Args:
        K: shape (N, N) - kernel matrix
        k: number of components
    
    Returns:
        projections: shape (N, k) - projected data in kernel feature space
    """
    ### WRITE YOUR SOLUTION HERE ###
    # 1. Center the kernel matrix
    # 2. Eigendecompose
    # 3. Select top-k and compute projections
    pass

""" END OF THIS PART """

In [ ]:
# Concentric circles data
N = 200
theta = np.random.uniform(0, 2 * np.pi, N)           # (200,)
r1 = 1 + 0.1 * np.random.randn(N // 2)               # inner
r2 = 3 + 0.1 * np.random.randn(N // 2)               # outer

X_inner = np.column_stack([r1 * np.cos(theta[:N//2]), r1 * np.sin(theta[:N//2])])  # (100, 2)
X_outer = np.column_stack([r2 * np.cos(theta[N//2:]), r2 * np.sin(theta[N//2:])])  # (100, 2)
X_circles = np.vstack([X_inner, X_outer])  # (200, 2)
labels = np.array([0] * (N//2) + [1] * (N//2))  # (200,)

# Linear PCA
X_centered = X_circles - X_circles.mean(axis=0)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
scores_linear = U[:, :1] * S[:1]  # (200, 1)

# Kernel PCA with RBF kernel
K = rbf_kernel(X_circles, sigma=1.0)      # (200, 200)
scores_kernel = kernel_pca(K, k=1)         # (200, 1)

print("Linear PCA first component:")
print(f"  Inner circle mean: {scores_linear[labels==0].mean():.4f}")
print(f"  Outer circle mean: {scores_linear[labels==1].mean():.4f}")
print(f"  Overlap: high (cannot separate)")

print("\nKernel PCA first component:")
print(f"  Inner circle mean: {scores_kernel[labels==0].mean():.4f}")
print(f"  Outer circle mean: {scores_kernel[labels==1].mean():.4f}")
print(f"  Separation: {abs(scores_kernel[labels==0].mean() - scores_kernel[labels==1].mean()):.4f}")